In [ ]:
from sctoolbox.utils.jupyter import bgcolor, _compare_version

# change the background of input cells
bgcolor("PowderBlue", select=[3, 5, 8, 10, 13])

nb_name = "Hotspot.ipynb"

_compare_version(nb_name)

# Hotspot
<hr style="border:2px solid black"> </hr>

## 1 - Description

# Hotspot Analysis

Hotspot identifies genes with informative local expression structure in single-cell data by computing gene-wise autocorrelations on a cell neighborhood graph. Based on these genes, local gene-gene correlations can be calculated and grouped into modules representing putative gene programs.
Hotspot documentation: https://hotspot.readthedocs.io/en/latest/

The aim of this notebook is to detect locally variable genes, calculate local gene-gene correlations, and group correlated genes into modules. These modules can represent putative gene programs that are active in specific cell states, developmental stages, or cell populations.

The main outputs of this notebook are Hotspot autocorrelation results, selected informative genes, local correlation matrices, gene modules, module scores per cell, summary tables, and visualizations such as UMAP plots and module-level summaries.

## Prerequisites

Requires a preprocessed AnnData object with raw counts stored in a layer, usually `adata.layers["raw"]`, and relevant metadata in `adata.obs` for plotting or subsetting.

This notebook requires a low-dimensional representation of the data to construct the cell neighborhood graph used by Hotspot. PCA and Harmony-corrected PCA can be computed within this notebook if needed. Other latent representations, such as scVI or Scanorama, must already be present in `adata.obsm` if they should be used for Hotspot.

Move this notebook into the notebook folder of the respective analysis, for example `rna_analysis/notebooks/`, before using it.


_______

## 2 - Setup

In [ ]:
import sctoolbox
import sctoolbox.utils as utils
import sctoolbox.tools.hotspot as hotspot

import pandas as pd
from IPython.display import display

sctoolbox.settings.settings_from_config("config.yaml", key="hotspot")

with pd.option_context("display.max.rows", None, "display.max_colwidth", None):
    display(sctoolbox.utils.general.get_version_report(report="versions.yml"))

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Set inpput anndata
clustered_adata = "anndata_4.h5ad"

_________

## 3 - Load anndata

In [ ]:
adata = utils.adata.load_h5ad(clustered_adata)

with pd.option_context("display.max.rows", 5, "display.max.columns", None):
    display(adata)
    display(adata.obs)
    display(adata.var)

## 4 -  Plot UMAP
<hr style="border:2px solid black"> </hr>

This section visualizes the full AnnData object before optional subsetting and Hotspot analysis. The global UMAP provides an overview of the dataset structure and shows how cells are distributed according to the selected condition column.

This helps users inspect sample, condition, batch, or cell population patterns before deciding whether to run Hotspot on the full dataset or on a biologically relevant subset. The UMAP is used for exploratory visualization and does not determine the latent representation used by Hotspot later in the notebook.


<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Global UMAP overview before subsetting

# Column in adata.obs used to color the UMAP
condition_column = "sample"

# Low-dimensional representation used to compute the UMAP if needed.
# Common options: "X_pca", "X_pca_harmony", "X_scVI", "X_scanorama".
global_umap_use_rep = "X_pca"

# If True, recompute the UMAP even if adata.obsm["X_umap"] already exists.
global_force_recompute_umap = False

# If True, recompute the representation used for UMAP, e.g. PCA or Harmony.
global_force_recompute_rep = False

# Controls how tightly cells are packed together in the UMAP.
# Smaller values create more compact clusters.
min_dist = 0.5

# Controls the overall spread of the UMAP embedding.
# Larger values place clusters farther apart.
spread = 1.0

# Number of UMAP dimensions to compute.
# Usually 2 for standard UMAP plots.
n_components = 2

# Random seed for reproducible UMAP results.
random_state = 0

# Optional plot title.
# If empty, the plotting function can use its default title.
title = ""

In [ ]:
hotspot.ensure_umap_for_visualization(
    adata,
    use_rep=global_umap_use_rep,
    batch_key=condition_column,
    is_subset=False,
    recompute_if_subset=False,
    force_recompute_umap=global_force_recompute_umap,
    force_recompute_rep=global_force_recompute_rep,
    neighbors_key="neighbors_global_umap",
    min_dist=min_dist,
    spread=spread,
    n_components=n_components,
    random_state=random_state,
)

_ = hotspot.plot_umap(
    adata,
    condition_column=condition_column,
    title=title,
    save="global_umap_overview.pdf",
    report="global_umap_overview.png",
)

__________

## 5 - Subset data (optional)
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# ============================================================
# Optional subsetting before running Hotspot
# If None the Anndata won't be changed
# Example:
# subset_filters = {
#     "sample": [3, 6],
#     "subcelltype": ["A+", "A-"],
# }
# ============================================================

subset_filters = {}
subset_copy = True
drop_unused_categories = False

In [ ]:
# Subset the celltype an early stage for example
adata_used = hotspot.subset_adata(
        adata=adata,
        subset_filters=subset_filters,
        copy=subset_copy,
        drop_unused_categories=drop_unused_categories,
    )
adata_used

In [ ]:
# Set subset true
subset_was_applied = hotspot.was_subsetted(
    original_adata=adata,
    current_adata=adata_used,
)

print(f"Subset was applied: {subset_was_applied}")

## 6 - Filter genes
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Gene detection filtering settings
GENE_DETECTION_FILTERING = {
    # Layer used for detection count.
    # Usually raw counts are preferred, e.g. "raw", "counts", or None for adata.X
    "layer_key": "raw",

    # Minimum number of cells in which a gene must be detected
    "min_cells": 50,

    # Only inspect thresholds, useful for deciding min_cells
    "thresholds_to_show": [10, 20, 30, 50, 100],

    # If False, only show statistics and plot.
    # If True, actually remove genes detected in fewer than min_cells cells.
    "filter_genes": False,

    # Plot histogram of gene detection frequency
    "plot": True,

    # Number of histogram bins
    "bins": 50,

    # Save detection counts into adata.var
    "add_to_var": True,

    # Name of the column added to adata.var
    "var_column": "n_cells_detected",

    # Return a copied filtered AnnData object
    "copy": True,
}

In [ ]:
adata_filtered, gene_detection, gene_filter_summary = hotspot.inspect_and_filter_genes_by_cell_detection(
    adata_used,
    **GENE_DETECTION_FILTERING,
)

## 7 - Select and calculate Hotspot latent key for hotspot
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Metadata columns
condition_column = "sample"      # for plotting / coloring
batch_key = "sample"             # for Harmony, if needed; set None if no batch correction

# Hotspot latent representation settings
HOTSPOT_REPRESENTATION = {
    # Options:
    # "auto"           -> use best available representation
    # "X_pca"          -> compute/use PCA
    # "X_pca_harmony"  -> compute/use Harmony-corrected PCA
    # "X_scVI"         -> use existing scVI representation if available
    # "X_scanorama"    -> use existing Scanorama representation if available
    "preferred_rep": "auto",

    # Priority if preferred_rep == "auto"
    "priority": [
        "X_pca_harmony",
        "X_scVI",
        "X_scanorama",
        "X_pca",
    ],

    # If no suitable representation exists, compute PCA
    "compute_pca_if_missing": True,

    # If batch_key is available and Harmony is wanted
    "compute_harmony_if_missing": True,

    # If the object was subsetted, recompute PCA/Harmony
    "recompute_if_subset": True,

    # Do not overwrite existing embeddings unless explicitly wanted
    "force_recompute_pca": False,
    "force_recompute_harmony": False,

    # External embeddings cannot be recomputed automatically.
    # If False, X_scVI / X_scanorama are skipped after subsetting.
    # If True, existing subsetted X_scVI / X_scanorama can still be used.
    "allow_existing_external_after_subset": False,
}

# PCA settings
PCA_SETTINGS = {
    "pca_key": "X_pca",
    "n_comps": 50,
    "use_highly_variable": False,
    "layer": None,
    "svd_solver": None,
    "random_state": 0,
}

# Harmony settings
HARMONY_SETTINGS = {
    "basis": "X_pca",
    "adjusted_basis": "X_pca_harmony",
    "batch_key": batch_key,
}

# Umap setting
subset_umap_use_rep = "X_pca"

In [ ]:
hotspot_latent_key = hotspot.ensure_hotspot_latent_representation(
    adata=adata_filtered,
    representation_settings=HOTSPOT_REPRESENTATION,
    pca_settings=PCA_SETTINGS,
    harmony_settings=HARMONY_SETTINGS,
    is_subset=subset_was_applied,
)

print(f"Selected Hotspot latent representation: {hotspot_latent_key}")

## 8 - Calculate UMAP after subsetting
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
batch_key = condition_column
recompute_if_subset = True,
force_recompute_umap = False,
force_recompute_rep = False,
neighbors_key = "neighbors_hotspot_umap",
min_dist = 0.5,
spread = 1.0,
n_components = 2,
use_highly_variable = False,
random_state = 0,

In [ ]:
hotspot.ensure_umap_for_visualization(
    adata_filtered,
    use_rep=subset_umap_use_rep,
    batch_key=batch_key,
    is_subset=subset_was_applied,
    recompute_if_subset=recompute_if_subset,
    force_recompute_umap=force_recompute_umap,
    force_recompute_rep=force_recompute_rep,
    neighbors_key=neighbors_key,
    min_dist=min_dist,
    spread=spread,
    n_components=n_components,
    use_highly_variable=use_highly_variable,
    random_state=random_state,
)

_ = hotspot.plot_umap(
    adata_filtered,
    condition_column=condition_column,
    title="",
    save="hotspot_subset_umap.pdf",
    report="hotspot_subset_umap.png",
)

## 9 - Hotspot Run
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

### 9.1 - input hotspot parameters

In [ ]:
# Input / output
output_dir = "hotspot_results"
output_prefix = "hotspot"
adata_output = "hotspot.h5ad"

# Hotspot input settings
layer_key = "raw"
latent_obsm_key = hotspot_latent_key
umi_counts_obs_key = "total_counts"
model = "danb"

# Graph settings
n_neighbors = 30
weighted_graph = False
jobs = 8

# Gene selection
gene_selection_method = "top_n_z"   # available: "fdr", "top_n_z"
autocorr_fdr_threshold = 0.05
top_n = 500

# Module settings
min_gene_threshold = 20
core_only = True
module_fdr_threshold = 0.05

# Outputs
compute_local_correlations = True
compute_modules = True
compute_module_scores = True

# Whether Hotspot results should be added to adata
attach_results_to_adata = True
module_prefix = "Hotspot module"
overwrite_attached_results = True

### 9.2 - Run hotspot

In [ ]:
results = hotspot.run_hotspot(
    adata=adata_filtered,
    layer_key=layer_key,
    latent_obsm_key=latent_obsm_key,
    umi_counts_obs_key=umi_counts_obs_key,
    model=model,
    n_neighbors=n_neighbors,
    weighted_graph=weighted_graph,
    jobs=jobs,
    top_n=top_n,
    gene_selection_method=gene_selection_method,
    autocorr_fdr_threshold=autocorr_fdr_threshold,
    min_gene_threshold=min_gene_threshold,
    core_only=core_only,
    module_fdr_threshold=module_fdr_threshold,
    compute_local_correlations=compute_local_correlations,
    compute_modules=compute_modules,
    compute_module_scores=compute_module_scores,
    attach_results_to_adata=attach_results_to_adata,
    module_prefix=module_prefix,
    overwrite_attached_results=overwrite_attached_results,
)

_________

## 10 - Results
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
fdr_threshold = 0.05
output_prefix = "hotspot"
store_local_correlations = False

In [ ]:
summary = hotspot.show_hotspot_results(
    adata=adata_filtered,
    results=results,
    fdr_threshold=fdr_threshold,
    output_prefix=output_prefix,
)

## 11 - Module enrichment
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
gene_sets = ["MSigDB_Hallmark_2020", "Reactome_2022",]
organism = "human"
min_module_size = 5
overwrite = True
inplace = True
save_table = "hotspot_module_enrichment.tsv"

# Plot enrichment
top_n = 5


### 11.1 - Run module enrichment

In [ ]:
hotspot.run_hotspot_module_enrichment(
    adata=adata_filtered,
    results=results,
    gene_sets=gene_sets,
    organism=organism,
    min_module_size=min_module_size,
    overwrite=overwrite,
    inplace=inplace,
    save_table=save_table,
)

### 11.2 - plot module enrichment

In [ ]:
# Plot module enrichment
hotspot.plot_hotspot_module_enrichment(
    adata_filtered,
    top_n=top_n,
    save="hotspot_module_enrichment.pdf",
    report="hotspot_module_enrichment.png"
)

## 12 - Save filtered adata
<hr style="border:2px solid black"> </hr>
Store the final results

In [ ]:
adata_filtered = hotspot.add_hotspot_results_to_adata(
    adata=adata_filtered,
    results=results,
    prefix=output_prefix,
    fdr_threshold=fdr_threshold,
    store_local_correlations=store_local_correlations,
)

In [ ]:
# Saving the data
adata_filtered = hotspot.stringify_adata_keys_for_saving(adata_filtered)
adata_output = "anndata_hotspot.h5ad"
utils.adata.save_h5ad(adata_filtered, adata_output, report=[
        "hotspot_overview.md",
        "hotspot_obs.png",
        "hotspot_var.png"])

In [ ]:
sctoolbox.settings.close_logfile()